In [ ]:
import random
import os
import numpy as np
import time
import torch
import torch.nn as nn 
import pandas as pd 
import argparse
import torch.optim as optim
import torch.utils.data as data
from tensorboardX import SummaryWriter
import scipy.sparse as sp
import torch.optim as optim
import torch.backends.cudnn as cudnn

参数设置：

In [ ]:
#设置伪随机数，方便实验结果复现
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # 即使没有 CUDA，也可以设置，不会产生影响
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # 在没有 CUDA 的情况下通常将其设置为 False

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("--seed", type=int, default=42,help="Seed")
parser.add_argument("--lr", type=float, default=0.01, help="learning rate")
parser.add_argument("--lamda", type=float, default=0.001, help="model regularization rate")
parser.add_argument("--batch_size", type=int, default=4096, help="batch size for training")
parser.add_argument("--epochs", type=int,default=50,  help="training epoches")
parser.add_argument("--top_k", type=int, default=10, help="compute metrics@top_k")
parser.add_argument("--factor_num", type=int,default=32, help="predictive factors numbers in the model")
parser.add_argument("--num_ng", type=int,default=4, help="sample negative items for training")
parser.add_argument("--test_num_ng", type=int,default=100, help="sample part of negative items for testing")
parser.add_argument("--out", default=True,help="save model or not")
parser.add_argument("--gpu", type=str,default="0",  help="gpu card ID")
args=parser.parse_known_args()[0]

args=parser.parse_known_args()[0]
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
writer = SummaryWriter()

seed_everything(args.seed)

处理数据：

In [ ]:
class BPRData(data.Dataset):
    def __init__(self, features):
        super(BPRData, self).__init__()
        self.features = pd.read_csv(features,sep='\t', names=['user_id', 'item_i', 'item_j'], engine='python')

    def __len__(self):
        return  len(self.features)
    
    def __getitem__(self, idx):
        features = self.features
        
        user = features.iloc[idx, 0]
        item_i = features.iloc[idx, 1]
        item_j = features.iloc[idx, 2] 
        
        return user, item_i, item_j 

In [ ]:
class BPR(nn.Module):
    def __init__(self, user_num, item_num, factor_num):
        super(BPR, self).__init__()
        """
        user_num: number of users;
        item_num: number of items;
        factor_num: number of predictive factors.
        """
        self.embed_user = nn.Embedding(user_num, factor_num)
        self.embed_item = nn.Embedding(item_num, factor_num)

        nn.init.normal_(self.embed_user.weight, std=0.01)
        nn.init.normal_(self.embed_item.weight, std=0.01)

    def forward(self, user, item_i, item_j):
        user = self.embed_user(user)
        item_i = self.embed_item(item_i)
        item_j = self.embed_item(item_j)

        prediction_i = (user * item_i).sum(dim=-1)
        prediction_j = (user * item_j).sum(dim=-1)
        return prediction_i, prediction_j

评估：

In [ ]:
def hit(gt_item, pred_items):
    if gt_item in pred_items:
        return 1
    return 0


def ndcg(gt_item, pred_items):
    if gt_item in pred_items:
        index = pred_items.index(gt_item)
        return np.reciprocal(np.log2(index+2))
    return 0


def metrics(model, test_loader, top_k):
    HR, NDCG = [], []

    for user, item_i, item_j in test_loader:
        user = user.to(device)
        item_i = item_i.to(device)
        item_j = item_j.to(device) # not useful when testing

        prediction_i, prediction_j = model(user, item_i, item_j)
        _, indices = torch.topk(prediction_i, top_k)
        recommends = torch.take(
                item_i, indices).cpu().numpy().tolist()

        gt_item = item_i[0].item()
        HR.append(hit(gt_item, recommends))
        NDCG.append(ndcg(gt_item, recommends))

    return np.mean(HR), np.mean(NDCG)

In [ ]:
n=3
for part in range(1,n+1):
    # paths
    main_path = f'./data/BPR_div_shard{n}/'

    train_data = main_path + f'sub_train{part}.csv'
    test_data = './data/BPR_div_train_and_test/test_data_nagetive.csv'

    model_path = './models1/'
    BPR_model_path = model_path + f'model_div{n}/div_shard{part}.pth'
    ############################## PREPARE DATASET ##########################

    # construct the train and test datasets
    train_dataset = BPRData(train_data)
    test_dataset = BPRData(test_data)

    train_loader = data.DataLoader(train_dataset,
            batch_size=args.batch_size, shuffle=True, num_workers=0)
    test_loader = data.DataLoader(test_dataset,
            batch_size=args.test_num_ng+1, shuffle=False, num_workers=0)

    user_num = 35761
    item_num = 1239
    ########################### CREATE MODEL #################################
    model = BPR(user_num, item_num, args.factor_num)
    model.to(device)

    optimizer = optim.SGD(
                model.parameters(), lr=args.lr, weight_decay=args.lamda)
    # writer = SummaryWriter() # for visualization

    ########################### TRAINING #####################################
    count, best_hr = 0, 0
    for epoch in range(args.epochs):
        model.train() 
        start_time = time.time()

        for user, item_i, item_j in train_loader:
            user = user.to(device)
            item_i = item_i.to(device)
            item_j = item_j.to(device)

            model.zero_grad()
            prediction_i, prediction_j = model(user, item_i, item_j)
            loss = - (prediction_i - prediction_j).sigmoid().log().sum()
            loss.backward()
            optimizer.step()
            # writer.add_scalar('data/loss', loss.item(), count)
            count += 1

        model.eval()
        HR, NDCG = metrics(model, test_loader, args.top_k)

        elapsed_time = time.time() - start_time
        print("The time elapse of epoch {:03d}".format(epoch) + " is: " + 
                time.strftime("%H: %M: %S", time.gmtime(elapsed_time)))
        print("HR: {:.5f}\tNDCG: {:.5f}".format(np.mean(HR), np.mean(NDCG)))

        if HR > best_hr:
            best_hr, best_ndcg, best_epoch = HR, NDCG, epoch
            if args.out:
                if not os.path.exists(model_path):
                    os.mkdir(model_path)
                torch.save(model, BPR_model_path)

    print("End. Best epoch {:03d}: HR = {:.5f}, \
        NDCG = {:.5f}".format(best_epoch, best_hr, best_ndcg))

The time elapse of epoch 000 is: 00: 03: 22
HR: 0.13001	NDCG: 0.06599
The time elapse of epoch 001 is: 00: 03: 41
HR: 0.25352	NDCG: 0.16985
The time elapse of epoch 002 is: 00: 03: 43
HR: 0.36284	NDCG: 0.24948
The time elapse of epoch 003 is: 00: 03: 39
HR: 0.41566	NDCG: 0.28132
The time elapse of epoch 004 is: 00: 03: 38
HR: 0.42592	NDCG: 0.28727
The time elapse of epoch 005 is: 00: 03: 28
HR: 0.42939	NDCG: 0.28854
The time elapse of epoch 006 is: 00: 03: 35
HR: 0.43188	NDCG: 0.28949
The time elapse of epoch 007 is: 00: 03: 46
HR: 0.43233	NDCG: 0.28993
The time elapse of epoch 008 is: 00: 03: 42
HR: 0.43258	NDCG: 0.28958
The time elapse of epoch 009 is: 00: 03: 41
HR: 0.43177	NDCG: 0.28911
The time elapse of epoch 010 is: 00: 03: 30
HR: 0.43101	NDCG: 0.28823
The time elapse of epoch 011 is: 00: 02: 55
HR: 0.43073	NDCG: 0.28786
The time elapse of epoch 012 is: 00: 02: 54
HR: 0.43115	NDCG: 0.28792
The time elapse of epoch 013 is: 00: 02: 54
HR: 0.43174	NDCG: 0.28779
The time elapse of e

The time elapse of epoch 016 is: 00: 02: 54
HR: 0.42329	NDCG: 0.28320
The time elapse of epoch 017 is: 00: 02: 54
HR: 0.42282	NDCG: 0.28271
The time elapse of epoch 018 is: 00: 02: 53
HR: 0.42257	NDCG: 0.28243
The time elapse of epoch 019 is: 00: 02: 54
HR: 0.42257	NDCG: 0.28213
The time elapse of epoch 020 is: 00: 02: 55
HR: 0.42327	NDCG: 0.28214
The time elapse of epoch 021 is: 00: 02: 53
HR: 0.42251	NDCG: 0.28177
The time elapse of epoch 022 is: 00: 02: 54
HR: 0.42310	NDCG: 0.28169
The time elapse of epoch 023 is: 00: 02: 53
HR: 0.42273	NDCG: 0.28122
The time elapse of epoch 024 is: 00: 02: 53
HR: 0.42338	NDCG: 0.28125
The time elapse of epoch 025 is: 00: 02: 55
HR: 0.42399	NDCG: 0.28104
The time elapse of epoch 026 is: 00: 02: 54
HR: 0.42355	NDCG: 0.28068
The time elapse of epoch 027 is: 00: 02: 54
HR: 0.42366	NDCG: 0.28034
The time elapse of epoch 028 is: 00: 02: 54
HR: 0.42324	NDCG: 0.28000
The time elapse of epoch 029 is: 00: 02: 54
HR: 0.42282	NDCG: 0.27967
The time elapse of e